In [ ]:
import json, textwrap

# ─── helpers ─────────────────────────────────────────────────────────────────
def md(src):
    return {"cell_type":"markdown","metadata":{},"source":[src.rstrip()]}

def code(src):
    lines = textwrap.dedent(src).splitlines(keepends=True)
    # ensure last line ends with \n
    if lines and not lines[-1].endswith('\n'):
        lines[-1] += '\n'
    return {"cell_type":"code","execution_count":None,"metadata":{},
            "outputs":[],"source":lines}

cells = []

# ═══════════════════════════════════════════════════════════════════════════════
# TITLE
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""# Predictive Modelling of Stillbirth and Neonatal Mortality in Sub-Saharan Africa
## Multicountry Analysis — Classical, Machine Learning and AI Models
**Victor et al. (LSHTM) | TRIPOD/PROBAST-aligned**

> **Como usar no Google Colab:**
> 1. Carregue este `.ipynb` via *File → Upload notebook*
> 2. Monte o Google Drive (Célula 1) e ajuste `DATA_PATH`
> 3. Execute em ordem: *Runtime → Run all*

---
### Estrutura do notebook
| Secção | Conteúdo |
|--------|----------|
| 0 | Instalação de dependências |
| 1 | Configuração & dados |
| 2 | Análise descritiva (Tabela 1, outcomes, QC, MNAR) |
| 3 | Pré-processamento (imputação, encoding, class imbalance, VIF) |
| 4 | Modelos — Família i: Regressão Logística (L1/L2/EN) |
| 5 | Modelos — Família ii: ML (RF, XGBoost, LightGBM, CatBoost) |
| 6 | Modelos — Família iii: AI/MLP (PyTorch + Optuna) |
| 7 | Métricas de performance (AUROC, AUPRC, calibração, DCA) |
| 8 | Validação interna — Nested CV + Bootstrap otimismo |
| 9 | Transportabilidade — IECV leave-one-country-out |
| 10 | Interpretabilidade — SHAP, permutação, PDP |
| 11 | Fairness — subgrupos, EOD, DPD |
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0 — INSTALL
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("## Célula 0 — Instalação de Dependências"))
cells.append(code("""
# Rodar apenas uma vez por sessão Colab
# (já instalados no ambiente de produção LSHTM)
!pip install xgboost lightgbm catboost optuna shap pyarrow fastparquet -q
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("## Célula 1 — Configuração, Drive & Imports"))
cells.append(code("""
# ── Google Drive ─────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Imports ───────────────────────────────────────────────────────────────────
import warnings, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from scipy.special import logit
from scipy.stats import binned_statistic, chi2_contingency

# sklearn
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.preprocessing   import StandardScaler, OneHotEncoder
from sklearn.linear_model    import LogisticRegression
from sklearn.ensemble        import RandomForestClassifier
from sklearn.impute          import SimpleImputer
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.inspection      import permutation_importance
from sklearn.metrics         import (
    roc_auc_score, average_precision_score, brier_score_loss,
    roc_curve, precision_recall_curve, confusion_matrix, f1_score
)

# ML
from xgboost  import XGBClassifier
from lightgbm import LGBMClassifier
try:
    from catboost import CatBoostClassifier; HAS_CAT = True
except ImportError:
    HAS_CAT = False; print("CatBoost não disponível")

# SMOTE (class imbalance)
try:
    from imblearn.over_sampling  import SMOTE
    from imblearn.pipeline       import Pipeline as ImbPipeline
    HAS_SMOTE = True
except ImportError:
    HAS_SMOTE = False; print("imbalanced-learn não disponível — usar class_weight")

import shap
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)
import torch
import torch.nn as nn

np.random.seed(42); torch.manual_seed(42)

# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_PATH = '/content/drive/MyDrive/df_ssa_from2010.parquet'   # ← ajustar
OUT = Path('/content/drive/MyDrive/outputs_analysis')
T   = OUT / 'tables'
F   = OUT / 'figures'
for d in [T/'desc', T/'model', F/'desc', F/'model']:
    d.mkdir(parents=True, exist_ok=True)

print('✓ Setup concluído')
print(f'  Saídas: {OUT}')
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — DATA DICTIONARY & LOAD
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 2 — Dicionário de Dados & Carregamento
> **Referência Paper §Methods — Data sources, Unit of analysis, Outcomes, Candidate predictors**
>
> Dataset: `df_ssa_from2010` — 141 variáveis harmonizadas de 7 estudos, 38 países SSA, 2010–2024
> - **Facility-based**: ALERT, PRECISE, PTBi, NCOPS, WHOMCS
> - **Population-based**: DHS (35 países), EN-INDEPTH
"""))
cells.append(code("""
# ── Dicionário completo — 141 variáveis (data_map_complete.csv) ───────────────

# Identificadores / Admin
ID_VAR    = 'unified_id'
STUDY_VAR = 'study_source'
YEAR_VAR  = 'studyyear'

# ── Desfechos (paper §Outcomes) ───────────────────────────────────────────────
# Primários
OUT_SB   = 'out_stillbirth'       # Stillbirth: morte fetal sem sinais de vida
OUT_NND  = 'out_nnd'              # Neonatal Death: óbito 0–28 dias
# Secundários
OUT_SB20 = 'out_stillbirth_20wks' # Stillbirth ≥20 sem
OUT_SB28 = 'out_stillbirth_28wks' # Stillbirth ≥28 sem (limiar internacional)
OUT_SBF  = 'out_fresh_stillbirth'
OUT_SBM  = 'out_macerated_stillbirth'
OUT_PD   = 'out_perinatal_death'  # SB + NND precoce
OUT_END  = 'out_nnd_early'        # NND precoce 0–7 dias
OUT_LND  = 'out_nnd_late'         # NND tardio 8–28 dias
OUT_LB   = 'out_livebirth'
# Desfechos neonatais
OUT_PRE  = 'out_preterm'          # <37 sem
OUT_VPRE = 'out_very_preterm'     # <32 sem
OUT_EPRE = 'out_extremely_preterm'# <28 sem
OUT_LBW  = 'out_lbw'              # <2500g
OUT_VLBW = 'out_vlbw'             # <1500g
OUT_ELBW = 'out_elbw'             # <1000g
OUT_SGA  = 'out_sga'
OUT_MULT = 'out_multiple'

ALL_OUTCOMES = {
    OUT_SB:   'Stillbirth',
    OUT_NND:  'Neonatal Death (0–28d)',
    OUT_PD:   'Perinatal Death',
    OUT_END:  'Early NND (0–7d)',
    OUT_LND:  'Late NND (8–28d)',
    OUT_SB28: 'Stillbirth ≥28 sem',
    OUT_SB20: 'Stillbirth ≥20 sem',
    OUT_SBF:  'Stillbirth Fresco',
    OUT_SBM:  'Stillbirth Macerado',
    OUT_PRE:  'Pré-termo (<37 sem)',
    OUT_VPRE: 'Muito Pré-termo (<32 sem)',
    OUT_EPRE: 'Extrem. Pré-termo (<28 sem)',
    OUT_LBW:  'Baixo Peso (<2500g)',
    OUT_VLBW: 'Muito Baixo Peso (<1500g)',
    OUT_SGA:  'Pequeno p/ Idade Gestacional',
    OUT_MULT: 'Gestação Múltipla',
}

# ── Preditores por domínio (paper §Candidate predictors — 7 domínios) ─────────
# (i) Sócio-demográfico materno
DOM_SOCIODEMO = ['mat_age','mat_age_cat','mat_marital_status','mat_religion',
                 'mat_ethnicity','mat_education','mat_literacy','mat_occupation',
                 'mat_urban_rural','mat_country']

# (ii) Domicílio / SES
DOM_HH = ['hh_wealth_quintile','hh_ses_binary','hh_size','hh_asset_score',
          'hh_house_floor','hh_house_wall','hh_cooking_fuel',
          'hh_electricity','hh_water_source','hh_sanitation','hh_mosquito_net',
          'hh_asset_radio','hh_asset_tv','hh_asset_mobile',
          'hh_asset_motorbike','hh_asset_car','hh_poverty_likelihood']

# (iii) História obstétrica
DOM_OBS = ['mat_gravidity','mat_parity','obs_previous_csection',
           'mat_previous_stillbirth']

# (iv) ANC
DOM_ANC = ['mat_anc_visits','anc_attendance','anc_tetanus']

# (v) Contextual / Ambiental — ERA5, CHIRPS, SRTM, ACAG PM2.5
DOM_ENV = ['env_latitude','env_longitude','env_elevation','env_slope',
           'env_temp_mean_delivery','env_humidity_delivery',
           'env_precipitation_delivery','env_heat_index_delivery',
           'env_pm25_annual',          # env_pm25_delivery excluído (r=1.0 colinear)
           'env_season_delivery','env_season_conception']

# (vi) Gravidez / Parto — cenário 2+
DOM_BIRTH = ['mat_delivery_mode','mat_delivery_location','mat_birth_attendant',
             'mat_csection','mat_obstructed_labour','mat_prom','out_multiple',
             'mat_height_cm','mat_weight_kg','mat_bmi',
             'mat_sbp','mat_dbp','mat_hypertension','mat_hypertension_stage',
             'mat_preeclampsia','mat_hiv_status','mat_syphilis',
             'preg_anaemia','life_tobacco','fat_education','fat_occupation']

# (vii) Neonatal — cenário 3+
DOM_NEO = ['out_ga_weeks','out_birthweight_g','out_infant_sex',
           'out_apgar_1min','out_apgar_5min','out_preterm',
           'neo_size_at_birth','out_lbw','out_sga']

# ── 4 Cenários clínicos (paper §Prediction scenarios) ────────────────────────
SCENARIOS = {
    'S1_Antenatal':    DOM_SOCIODEMO + DOM_HH + DOM_OBS + DOM_ANC + DOM_ENV,
    'S2_Admission':    DOM_SOCIODEMO + DOM_HH + DOM_OBS + DOM_ANC + DOM_ENV + DOM_BIRTH,
    'S3_PostDelivery': DOM_SOCIODEMO + DOM_HH + DOM_OBS + DOM_ANC + DOM_ENV + DOM_BIRTH + DOM_NEO,
}
# Cenário 4 (tempo-ao-evento) — Cox PH — ver Célula 6

# ── 38 Países SSA (country_distribution.csv) ──────────────────────────────────
ALL_COUNTRIES = [
    'Nigeria','Senegal','Uganda','Malawi','Kenya','Ethiopia',
    'Congo Democratic Republic','Tanzania','Zambia','Benin','Mali',
    'Ghana','Burkina Faso','Rwanda','Angola','Sierra Leone',
    'Cameroon','Mozambique','Burundi',"Cote d'Ivoire",'Chad',
    'Gambia','Guinea','Liberia','Niger','Madagascar','Gabon',
    'Zimbabwe','Mauritania','Congo','Guinea-Bissau','Togo',
    'Lesotho','Namibia','South Africa','Comoros',
    'Dem Rep Of The Congo','The Gambia',
]

# ── Estudos e design ──────────────────────────────────────────────────────────
STUDY_DESIGN = {
    'DHS':        'Population-based survey (35 países SSA)',
    'EN-INDEPTH': 'Population-based HDSS (Ethiopia, Ghana, Guinea-Bissau, Uganda)',
    'ALERT':      'Facility-based stepped-wedge RCT (Uganda, Malawi, Benin, Tanzania)',
    'PTBi':       'Facility-based cluster RCT (Kenya, Uganda)',
    'WHOMCS':     'Facility-based registry (29 países; 6 SSA)',
    'PRECISE':    'Facility-based prospective cohort (Gambia, Kenya, Mozambique)',
    'NCOPS':      'Facility-based neonatal ICU cohort (Uganda)',
}

# ── Variáveis contínuas para sumário e correlação ────────────────────────────
CONT_VARS = [
    'mat_age','mat_gravidity','mat_parity',
    'mat_height_cm','mat_weight_kg','mat_bmi','mat_muac',
    'mat_sbp','mat_dbp','mat_anc_visits','anc_tetanus',
    'hh_size','hh_asset_score','hh_poverty_likelihood',
    'out_ga_weeks','out_birthweight_g',
    'out_apgar_1min','out_apgar_5min','out_ageatdeath',
    'env_latitude','env_longitude','env_elevation','env_slope',
    'env_temp_mean_delivery','env_humidity_delivery',
    'env_precipitation_delivery','env_heat_index_delivery',
    'env_pm25_annual','env_pm25_delivery',
]

# ── Helper: Yes/1/True → int binário ─────────────────────────────────────────
def to_bin(col):
    return col.astype(str).str.strip().str.lower().isin(
        ['1','true','yes','y']).astype(int)

# ── Carregar dataset ──────────────────────────────────────────────────────────
print('Carregando dataset...')
df = pd.read_parquet(DATA_PATH)
df.columns = df.columns.str.strip().str.lower()
N = len(df)
print(f'  ✓ {N:,} registros × {df.shape[1]} variáveis')
print(f'  ✓ Período  : {int(df[YEAR_VAR].min())}–{int(df[YEAR_VAR].max())}')
print(f'  ✓ Países   : {df["mat_country"].nunique()} ({df["mat_country"].unique().tolist()})')
print(f'  ✓ Estudos  : {df[STUDY_VAR].value_counts().to_dict()}')
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — DESCRIPTIVE ANALYSIS
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 3 — Análise Descritiva
> **Paper §Data sources / Unit of analysis / Outcomes**
> Tabela 1 do manuscrito: características da amostra, prevalências, distribuições por estudo e país.
"""))
cells.append(code("""
# ═══════════════════════════════════════════════════════════════════════════════
# 3.1 VISÃO GERAL DO DATASET
# ═══════════════════════════════════════════════════════════════════════════════
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(exclude=np.number).columns.tolist()

overview = pd.DataFrame({
    'Métrica': ['Total registros','Total variáveis','Numéricas',
                'Categóricas/Factor','100% completas','>50% faltantes',
                'unified_id únicos','Linhas duplicadas',
                'Países únicos','Estudos únicos'],
    'Valor':   [f'{N:,}', df.shape[1], len(num_cols), len(cat_cols),
                int((df.isna().mean()==0).sum()),
                int((df.isna().mean()>0.5).sum()),
                df[ID_VAR].nunique(), int(df.duplicated().sum()),
                df['mat_country'].nunique(), df[STUDY_VAR].nunique()]
})
overview.to_csv(T/'desc/01_dataset_overview.csv', index=False)
print(overview.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# 3.2 DISTRIBUIÇÃO POR ESTUDO
# ═══════════════════════════════════════════════════════════════════════════════
std = df[STUDY_VAR].value_counts().rename_axis('study').reset_index(name='n')
std['pct']    = (std['n'] / N * 100).round(2)
std['design'] = std['study'].map(STUDY_DESIGN).fillna('—')
std.to_csv(T/'desc/02_study_distribution.csv', index=False)
print(f"\\n[3.2] Distribuição por estudo:\\n{std.to_string(index=False)}")

fig, ax = plt.subplots(figsize=(11,5))
colors = plt.cm.Set2(np.linspace(0,1,len(std)))
bars = ax.barh(std['study'], std['n'], color=colors, edgecolor='white', height=0.6)
for b, n, p in zip(bars, std['n'], std['pct']):
    ax.text(b.get_width()*1.01, b.get_y()+b.get_height()/2,
            f'{n:,}  ({p}%)', va='center', fontsize=10, fontweight='bold')
ax.set_xlabel('N registros', fontweight='bold')
ax.set_title('Registros por Fonte de Estudo — df_ssa_from2010 (n=3.5M)',
             fontsize=13, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.set_xlim(0, std['n'].max()*1.25)
plt.tight_layout(); plt.savefig(F/'desc/study_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 3.3 DISTRIBUIÇÃO POR PAÍS (todos os 38)
# ═══════════════════════════════════════════════════════════════════════════════
ctry = df['mat_country'].value_counts().rename_axis('country').reset_index(name='n')
ctry['pct'] = (ctry['n'] / N * 100).round(2)
ctry.to_csv(T/'desc/03_country_distribution.csv', index=False)
print(f"\\n[3.3] Países ({len(ctry)}):")
print(ctry.to_string(index=False))

fig, ax = plt.subplots(figsize=(10,13))
ax.barh(ctry['country'][::-1], ctry['n'][::-1],
        color=plt.cm.tab20(np.linspace(0,1,len(ctry))), height=0.7)
ax.set_xlabel('N registros', fontweight='bold')
ax.set_title('Registros por País — df_ssa_from2010 (38 países SSA)',
             fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
plt.tight_layout(); plt.savefig(F/'desc/country_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 3.4 DISTRIBUIÇÃO TEMPORAL
# ═══════════════════════════════════════════════════════════════════════════════
yr_dist = (pd.to_numeric(df[YEAR_VAR], errors='coerce').dropna().astype(int)
           .value_counts().sort_index().reset_index())
yr_dist.columns = ['year','n']
yr_dist.to_csv(T/'desc/04_year_distribution.csv', index=False)

fig, ax = plt.subplots(figsize=(10,4))
ax.fill_between(yr_dist['year'], yr_dist['n'], alpha=0.25, color='#1B3A6B')
ax.plot(yr_dist['year'], yr_dist['n'], color='#1B3A6B', lw=2, marker='o', ms=5)
ax.set_xlabel('Ano', fontweight='bold'); ax.set_ylabel('N registros', fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{int(x):,}'))
ax.set_title('Distribuição Temporal — df_ssa_from2010 (2010–2024)', fontweight='bold')
plt.tight_layout(); plt.savefig(F/'desc/year_distribution.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 3.5 PREVALÊNCIA DE DESFECHOS (paper §Outcomes — Tabela 1)
# ═══════════════════════════════════════════════════════════════════════════════
print("\\n[3.5] Prevalência dos desfechos:")
out_rows = []
for col, lbl in ALL_OUTCOMES.items():
    if col not in df.columns: continue
    ev = to_bin(df[col]).sum(); dn = df[col].notna().sum()
    out_rows.append({'outcome': col, 'label': lbl, 'events': int(ev),
                     'denom': int(dn), 'miss': int(N-dn),
                     'prev_%': round(ev/dn*100,3) if dn>0 else None,
                     'rate_1000': round(ev/dn*1000,2) if dn>0 else None})
out_prev = pd.DataFrame(out_rows)
out_prev.to_csv(T/'desc/05_outcomes_overall.csv', index=False)
print(out_prev[['label','events','denom','miss','prev_%','rate_1000']].to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# 3.6 PREVALÊNCIA POR ESTUDO + FOREST PLOT
# ═══════════════════════════════════════════════════════════════════════════════
rows_s = []
for st, g in df.groupby(STUDY_VAR):
    for col in [c for c in ALL_OUTCOMES if c in df.columns]:
        ev = to_bin(g[col]).sum(); dn = g[col].notna().sum()
        rows_s.append({'study':st,'outcome':col,'label':ALL_OUTCOMES[col],
                       'n_study':len(g),'events':int(ev),'denom':int(dn),
                       'rate_1000':round(ev/dn*1000,2) if dn>0 else None})
oc_by_study = pd.DataFrame(rows_s)
oc_by_study.to_csv(T/'desc/06_outcomes_by_study.csv', index=False)

key_oc = [c for c in [OUT_SB,OUT_NND,OUT_PD,OUT_PRE,OUT_LBW] if c in df.columns]
fig, axes = plt.subplots(1, len(key_oc), figsize=(5*len(key_oc),6), sharey=True)
if len(key_oc)==1: axes=[axes]
for ax, oc in zip(axes, key_oc):
    sub = oc_by_study[oc_by_study['outcome']==oc].sort_values('rate_1000')
    ax.scatter(sub['rate_1000'], sub['study'],
               s=sub['n_study']/sub['n_study'].max()*200+40, color='#1B3A6B', alpha=0.85)
    ov = out_prev.loc[out_prev['outcome']==oc,'rate_1000']
    if not ov.empty:
        ax.axvline(ov.values[0], color='red', lw=1.5, ls='--',
                   label=f'Overall: {ov.values[0]:.0f}')
    ax.set_xlabel('Taxa por 1.000', fontsize=9, fontweight='bold')
    ax.set_title(ALL_OUTCOMES[oc], fontsize=9, fontweight='bold')
    ax.grid(axis='x', alpha=0.35); ax.legend(fontsize=8)
plt.suptitle('Taxas de Desfechos por Estudo — df_ssa_from2010', fontweight='bold')
plt.tight_layout(); plt.savefig(F/'desc/forest_outcomes_by_study.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 3.7 PREVALÊNCIA POR PAÍS (todos os 38)
# ═══════════════════════════════════════════════════════════════════════════════
rows_c = []
for oc in [OUT_SB, OUT_NND, OUT_PD]:
    if oc not in df.columns: continue
    for country, g in df.groupby('mat_country'):
        ev = to_bin(g[oc]).sum(); dn = g[oc].notna().sum()
        rows_c.append({'country':country,'outcome':oc,'n':len(g),
                       'events':int(ev),'denom':int(dn),
                       'rate_1000':round(ev/dn*1000,2) if dn>0 else None})
pd.DataFrame(rows_c).to_csv(T/'desc/07_outcomes_by_country.csv', index=False)
print("  ✓ 07_outcomes_by_country.csv salvo (38 países × 3 outcomes)")

# ═══════════════════════════════════════════════════════════════════════════════
# 3.8 SUMÁRIO PREDITORES CONTÍNUOS (paper §Tabela 1 — contínuos)
# ═══════════════════════════════════════════════════════════════════════════════
cont_p = [v for v in CONT_VARS if v in df.columns]
cs = df[cont_p].describe(percentiles=[.05,.25,.50,.75,.95]).T
cs.index.name = 'variable'
cs['n_missing']   = df[cont_p].isna().sum().values
cs['pct_missing'] = (df[cont_p].isna().mean()*100).round(2).values
cs = cs.rename(columns={'50%':'median','25%':'q25','75%':'q75','5%':'p05','95%':'p95'})
cs.round(3).to_csv(T/'desc/08_continuous_summary.csv')
print(f"\\n[3.8] Preditores contínuos ({len(cont_p)} vars):")
print(cs[['count','mean','std','median','q25','q75','pct_missing']].round(2).to_string())

# ═══════════════════════════════════════════════════════════════════════════════
# 3.9 SUMÁRIO PREDITORES CATEGÓRICOS
# ═══════════════════════════════════════════════════════════════════════════════
ALL_CAT = [
    'study_source','study_design','module',
    'mat_country','mat_urban_rural','mat_age_cat',
    'mat_marital_status','mat_religion','mat_ethnicity',
    'mat_education','mat_literacy','mat_occupation',
    'mat_hypertension','mat_hypertension_stage','mat_preeclampsia',
    'mat_eclampsia','mat_aph','mat_diabetes','mat_malaria','mat_anaemia',
    'mat_hiv_status','mat_syphilis','obs_previous_csection',
    'mat_previous_stillbirth','mat_delivery_mode','mat_delivery_location',
    'mat_birth_attendant','mat_csection','mat_prolonged_labour',
    'mat_obstructed_labour','mat_prom','mat_anc_provider',
    'hh_wealth_quintile','hh_ses_binary','hh_house_floor','hh_house_wall',
    'hh_ppi_band','hh_cooking_fuel','hh_heating_fuel','hh_lighting',
    'hh_electricity','hh_water_source','hh_sanitation','hh_mosquito_net',
    'hh_asset_radio','hh_asset_tv','hh_asset_mobile','hh_asset_motorbike','hh_asset_car',
    'anc_attendance','fat_education','fat_occupation',
    'out_infant_sex','out_multiple','out_ga_method','out_dob_source',
    'out_sga','out_lga','out_aga','out_sizefor_ga',
    'out_lbw','out_vlbw','out_elbw',
    'out_preterm','out_very_preterm','out_extremely_preterm',
    'out_fresh_stillbirth','out_macerated_stillbirth',
    'out_stillbirth','out_livebirth','out_nnd','out_perinatal_death',
    'neo_size_at_birth','preg_anaemia','preg_hiv','life_tobacco',
    'env_season_delivery','env_season_conception',
    'coordinate_source','loc_facility_type',
]
cat_p = [v for v in ALL_CAT if v in df.columns]
cat_rows = []
for v in cat_p:
    for level, n in df[v].value_counts(dropna=False).items():
        cat_rows.append({'variable':v,'level':str(level),'n':int(n),'pct':round(n/N*100,2)})
pd.DataFrame(cat_rows).to_csv(T/'desc/09_categorical_summary.csv', index=False)
print(f"  ✓ Categóricas: {len(cat_p)} variáveis → 09_categorical_summary.csv")

# ═══════════════════════════════════════════════════════════════════════════════
# 3.10 COMPLETUDE POR ESTUDO — MNAR ESTRUTURAL (paper §Missing data)
# ═══════════════════════════════════════════════════════════════════════════════
KEY_MISS = ['out_stillbirth','out_nnd','out_perinatal_death',
            'out_birthweight_g','out_ga_weeks','out_preterm','out_lbw',
            'mat_age','mat_parity','mat_height_cm','mat_weight_kg',
            'mat_anc_visits','mat_previous_stillbirth',
            'mat_hypertension','mat_hiv_status',
            'env_temp_mean_delivery','env_pm25_annual',
            'out_apgar_1min','out_apgar_5min']
kv = [v for v in KEY_MISS if v in df.columns]
comp = df.groupby(STUDY_VAR)[kv].apply(lambda g: (g.notna().mean()*100).round(1))
comp.to_csv(T/'desc/10_completeness_by_study.csv')
print(f"\\n[3.10] Completude (%) por estudo:\\n{comp.to_string()}")

fig, ax = plt.subplots(figsize=(max(12,len(kv)*0.85), 4.5))
sns.heatmap(comp, annot=True, fmt='.0f', cmap='RdYlGn',
            vmin=0, vmax=100, ax=ax, linewidths=0.4,
            cbar_kws={'label':'% Completo'})
ax.set_title('Completude de Variáveis-Chave por Estudo (MNAR estrutural)',
             fontweight='bold', fontsize=11)
plt.xticks(rotation=35, ha='right', fontsize=8)
plt.tight_layout(); plt.savefig(F/'desc/completeness_heatmap.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 3.11 MISSINGNESS GLOBAL (141 variáveis)
# ═══════════════════════════════════════════════════════════════════════════════
miss = pd.DataFrame({
    'variable': df.columns,
    'n_missing': df.isna().sum().values,
    'pct_missing': (df.isna().mean()*100).round(2).values
}).sort_values('pct_missing', ascending=False)
miss['domain'] = miss['variable'].apply(lambda v:
    'Outcomes' if v.startswith('out_') else 'Maternal' if v.startswith('mat_') else
    'Household/SES' if v.startswith('hh_') else 'Environmental' if v.startswith('env_') else
    'ANC' if v.startswith('anc_') else 'Obstetric' if v.startswith('obs_') else
    'Neonatal' if v.startswith('neo_') else 'Paternal' if v.startswith('fat_') else
    'Pregnancy' if v.startswith('preg_') else 'Lifestyle' if v.startswith('life_') else
    'Location' if v.startswith('loc_') else 'Admin/Other')
miss.to_csv(T/'desc/11_missingness_all141.csv', index=False)

top30 = miss.head(30)
fig, ax = plt.subplots(figsize=(9,9))
cols_m = ['#B52020' if p>=100 else '#C07820' if p>50 else '#254E96'
          for p in top30['pct_missing']]
ax.barh(top30['variable'][::-1], top30['pct_missing'][::-1],
        color=cols_m[::-1], edgecolor='white', height=0.7)
ax.axvline(50, color='#C07820', lw=1.5, ls='--', label='50%')
ax.axvline(90, color='#B52020', lw=1.5, ls='--', label='90% (MNAR estrutural)')
ax.set_xlabel('% Faltante', fontweight='bold')
ax.set_title('Top 30 Variáveis — Missingness Global (n=141)', fontweight='bold')
ax.legend()
plt.tight_layout(); plt.savefig(F/'desc/top30_missingness.png', dpi=200, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# 3.12 QC CHECKS (paper §Data harmonization and quality assurance)
# Regras: range, plausibilidade, consistência lógica, sentinel values
# ═══════════════════════════════════════════════════════════════════════════════
print("\\n[3.12] QC checks (range + lógica):")
qc_rules = [
    ('out_birthweight_g', 'BW < 300g (implausível — crítico)',    lambda x: x < 300),
    ('out_birthweight_g', 'BW > 6000g',                           lambda x: x > 6000),
    ('mat_age',           'Idade mãe < 10 anos (crítico)',         lambda x: x < 10),
    ('mat_age',           'Idade mãe > 55 anos',                   lambda x: x > 55),
    ('out_ga_weeks',      'GA < 20 semanas',                       lambda x: x < 20),
    ('out_ga_weeks',      'GA > 45 semanas',                       lambda x: x > 45),
    ('mat_height_cm',     'Altura < 100 cm',                       lambda x: x < 100),
    ('mat_weight_kg',     'Peso < 20 kg',                          lambda x: x < 20),
    ('mat_bmi',           'BMI < 10 ou > 70',                      lambda x: (x<10)|(x>70)),
    ('mat_sbp',           'SBP < 60 ou > 250 mmHg',                lambda x: (x<60)|(x>250)),
    ('mat_dbp',           'DBP < 30 ou > 150 mmHg',                lambda x: (x<30)|(x>150)),
    ('out_apgar_1min',    'Apgar 1min fora 0–10',                  lambda x: (x<0)|(x>10)),
    ('out_apgar_5min',    'Apgar 5min fora 0–10',                  lambda x: (x<0)|(x>10)),
    ('env_pm25_annual',   'PM2.5 = −999 (sentinel value → NA)',    lambda x: x < 0),
    ('mat_parity',        'Paridade < 0',                          lambda x: x < 0),
    ('mat_gravidity',     'Gravididade < 0',                       lambda x: x < 0),
]
qc_out = []
for var, issue, fn in qc_rules:
    if var not in df.columns: continue
    x = pd.to_numeric(df[var], errors='coerce')
    n = int(fn(x).sum())
    qc_out.append({'variable':var, 'issue':issue,
                   'n_violations':n, 'pct':round(n/N*100,4)})

# Lógica: Stillbirth=Yes E Livebirth=Yes (impossível)
if 'out_stillbirth' in df.columns and 'out_livebirth' in df.columns:
    both = (to_bin(df['out_stillbirth']) & to_bin(df['out_livebirth'])).sum()
    qc_out.append({'variable':'SB+LB','issue':'Stillbirth=Yes E Livebirth=Yes',
                   'n_violations':int(both),'pct':round(both/N*100,4)})

# GA<28sem mas BW>2000g
if 'out_ga_weeks' in df.columns and 'out_birthweight_g' in df.columns:
    ga = pd.to_numeric(df['out_ga_weeks'], errors='coerce')
    bw = pd.to_numeric(df['out_birthweight_g'], errors='coerce')
    impl = ((ga < 28) & (bw > 2000)).sum()
    qc_out.append({'variable':'GA+BW','issue':'GA<28sem mas BW>2000g',
                   'n_violations':int(impl),'pct':round(impl/N*100,4)})

qc_df = pd.DataFrame(qc_out)
qc_df.to_csv(T/'desc/12_qc_checks.csv', index=False)
print(qc_df.to_string(index=False))

# ═══════════════════════════════════════════════════════════════════════════════
# 3.13 CORRELAÇÃO E MULTICOLINEARIDADE (paper §Feature engineering — VIF)
# ═══════════════════════════════════════════════════════════════════════════════
num_corr = [v for v in CONT_VARS if v in df.columns and df[v].notna().sum() > N*0.05]
if len(num_corr) >= 3:
    corr = df[num_corr].corr(method='pearson')
    corr.to_csv(T/'desc/13_correlation_matrix.csv')

    coll = [(corr.columns[i], corr.columns[j], round(corr.iloc[i,j],3))
            for i in range(len(corr)) for j in range(i+1,len(corr))
            if abs(corr.iloc[i,j]) > 0.80]
    pd.DataFrame(coll, columns=['var1','var2','r'])\
      .to_csv(T/'desc/14_collinear_pairs_r080.csv', index=False)
    print(f"\\n[3.13] Pares colineares (|r|>0.80): {len(coll)}")
    for v1,v2,r in coll: print(f"    {v1}  ↔  {v2}  r={r}")

    fig, ax = plt.subplots(figsize=(max(10,len(num_corr)*0.85), max(9,len(num_corr)*0.75)))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
                center=0, vmin=-1, vmax=1, ax=ax, linewidths=0.4,
                annot_kws={'size':6})
    ax.set_title('Correlação de Pearson — Preditores Numéricos (paper §VIF/multicolinearidade)',
                 fontweight='bold')
    plt.xticks(rotation=45, ha='right', fontsize=7); plt.yticks(fontsize=7)
    plt.tight_layout(); plt.savefig(F/'desc/correlation_matrix.png', dpi=200, bbox_inches='tight')
    plt.show()

print("\\n✅ Secção 3 — Análise descritiva concluída.")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — PRE-PROCESSING
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 4 — Pré-Processamento
> **Paper §Missing data / Class imbalance / Feature engineering**
>
> - **MICE** (R/Python): imputação múltipla para regressão clássica  
> - **Median/mode + missingness flags**: ML pipeline (prevenir leakage)  
> - **SMOTE**: aplicado *apenas dentro* dos folds de treino  
> - **VIF**: multicolinearidade ≥ threshold → remoção
"""))
cells.append(code("""
# ── Configurar target e cenário aqui ─────────────────────────────────────────
TARGET   = 'out_stillbirth'   # 'out_nnd' | 'out_perinatal_death'
SCENARIO = 'S1_Antenatal'     # 'S2_Admission' | 'S3_PostDelivery'
# ─────────────────────────────────────────────────────────────────────────────

if TARGET not in df.columns:
    raise ValueError(f"Coluna '{TARGET}' não encontrada.")

# Cohort: registros com outcome definido
df_m       = df.copy()
df_m['_y'] = to_bin(df_m[TARGET])
df_m       = df_m[df_m[TARGET].notna()].reset_index(drop=True)
y          = df_m['_y'].values
pos_n      = int(y.sum())
scale_w    = (len(y) - pos_n) / pos_n    # para XGB scale_pos_weight

print(f"Target   : {TARGET}  |  Cenário: {SCENARIO}")
print(f"Registros: {len(df_m):,}  |  Eventos: {pos_n:,} ({y.mean()*100:.3f}%)")
print(f"Rácio neg/pos (scale_pos_weight): {scale_w:.1f}")

# Preditores do cenário
feats    = [f for f in SCENARIOS[SCENARIO] if f in df_m.columns]
X        = df_m[feats]
num_f    = X.select_dtypes(include=np.number).columns.tolist()
cat_f    = [c for c in feats if c not in num_f]
print(f"\\nPreditores: {len(feats)}/{len(SCENARIOS[SCENARIO])}  "
      f"(num={len(num_f)}, cat={len(cat_f)})")
print(f"Ausentes  : {set(SCENARIOS[SCENARIO]) - set(feats)}")

# ── Remoção de variáveis 100% ausentes (paper §Pre-modelling checklist) ───────
pct_miss_feat = {c: df_m[c].isna().mean() for c in feats}
feats_100miss = [c for c,p in pct_miss_feat.items() if p == 1.0]
if feats_100miss:
    print(f"\\n  Removendo {len(feats_100miss)} variáveis 100% ausentes: {feats_100miss}")
    feats = [f for f in feats if f not in feats_100miss]
    X = df_m[feats]
    num_f = X.select_dtypes(include=np.number).columns.tolist()
    cat_f = [c for c in feats if c not in num_f]

# ── Missingness indicators (paper §Missing data — ML pipeline) ────────────────
# Criados apenas para vars com >2% ausentes
miss_flags = []
for c in num_f:
    if X[c].isna().mean() > 0.02:
        df_m[f'_miss_{c}'] = X[c].isna().astype(int)
        miss_flags.append(f'_miss_{c}')
print(f"  Missingness flags: {len(miss_flags)}")

X = df_m[feats + miss_flags]

# ── Preprocessador (paper §Class imbalance and data transformation) ───────────
# z-score normalização (numéricas) + OneHot (categóricas)
num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('scl', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='most_frequent')),
                     ('enc', OneHotEncoder(handle_unknown='ignore',
                                           sparse_output=False))])
preprocessor = ColumnTransformer([
    ('num',  num_pipe, num_f),
    ('cat',  cat_pipe, cat_f),
    ('flag', 'passthrough', miss_flags),
], remainder='drop')

# ── Verificar pares colineares nos preditores do cenário ─────────────────────
num_feats_present = [v for v in num_f if df_m[v].notna().sum() > len(df_m)*0.05]
if len(num_feats_present) >= 2:
    corr_feats = df_m[num_feats_present].corr().abs()
    coll_feats = [(corr_feats.columns[i], corr_feats.columns[j],
                   round(corr_feats.iloc[i,j],3))
                  for i in range(len(corr_feats))
                  for j in range(i+1,len(corr_feats))
                  if corr_feats.iloc[i,j] > 0.80]
    if coll_feats:
        print(f"\\n  ⚠ Pares colineares nos preditores do cenário (|r|>0.80):")
        for v1,v2,r in coll_feats: print(f"    {v1} ↔ {v2}  r={r}")
        print("  → Ver paper §Feature engineering: remover um de cada par (VIF > 5/10)")

print("\\n✓ Pré-processamento configurado.")
print(f"  X shape: {X.shape}  |  y: {y.sum()} pos / {(1-y).sum()} neg")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — FAMILY i: LOGISTIC REGRESSION
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 5 — Família i: Regressão Logística (L1 / L2 / Elastic Net)
> **Paper §Classical statistical models**
> Modelos de regressão logística com penalização L1 (LASSO), L2 (Ridge) e Elastic Net
> como benchmark principal interpretável (TRIPOD: modelo de referência clínica).
> GEE e Cox PH (Cenário 4) no final desta célula.
"""))
cells.append(code("""
# ── Definição dos modelos Família i ───────────────────────────────────────────
LR_MODELS = {
    'LR_Ridge': Pipeline([
        ('prep', preprocessor),
        ('clf',  LogisticRegression(penalty='l2', C=0.1, solver='lbfgs',
                                    class_weight='balanced',
                                    max_iter=500, random_state=42)),
    ]),
    'LR_LASSO': Pipeline([
        ('prep', preprocessor),
        ('clf',  LogisticRegression(penalty='l1', C=0.1, solver='liblinear',
                                    class_weight='balanced',
                                    max_iter=500, random_state=42)),
    ]),
    'LR_ElasticNet': Pipeline([
        ('prep', preprocessor),
        ('clf',  LogisticRegression(penalty='elasticnet', C=0.1,
                                    l1_ratio=0.5, solver='saga',
                                    class_weight='balanced',
                                    max_iter=1000, random_state=42)),
    ]),
}

# ── Nested CV — 5 folds externos (paper §Hyperparameter tuning) ───────────────
# CV folds estratificados por outcome; em dados clusterizados (ALERT/PTBi),
# atribuir facilities inteiras aos folds (aqui: estratificação por country como proxy)
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
lr_cv_results = []

for name, pipe in LR_MODELS.items():
    aurocs, auprcs, briers = [], [], []
    for tr, te in SKF.split(X, y):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y[tr], y[te]
        pipe.fit(X_tr, y_tr)
        yp = pipe.predict_proba(X_te)[:,1]
        aurocs.append(roc_auc_score(y_te, yp))
        auprcs.append(average_precision_score(y_te, yp))
        briers.append(brier_score_loss(y_te, yp))
    lr_cv_results.append({
        'model': name, 'family': 'LR',
        'target': TARGET, 'scenario': SCENARIO,
        'auroc_mean': round(np.mean(aurocs),4), 'auroc_sd': round(np.std(aurocs),4),
        'auprc_mean': round(np.mean(auprcs),4), 'auprc_sd': round(np.std(auprcs),4),
        'brier_mean': round(np.mean(briers),5), 'brier_sd': round(np.std(briers),5),
    })
    print(f"  {name:18s}: AUROC={np.mean(aurocs):.3f}±{np.std(aurocs):.3f}  "
          f"AUPRC={np.mean(auprcs):.3f}±{np.std(auprcs):.3f}  "
          f"Brier={np.mean(briers):.4f}")

lr_cv_df = pd.DataFrame(lr_cv_results)
lr_cv_df.to_csv(T/'model/01a_cv_LR.csv', index=False)

# ── Fit final e coeficientes (paper: OR + IC) ─────────────────────────────────
lr_fitted = {}
for name, pipe in LR_MODELS.items():
    pipe.fit(X, y)
    lr_fitted[name] = pipe

# Extrair coeficientes do LR_Ridge (benchmark principal)
prep_fit = lr_fitted['LR_Ridge'].named_steps['prep']
lr_clf   = lr_fitted['LR_Ridge'].named_steps['clf']
try:
    ohe_n = prep_fit.named_transformers_['cat']['enc'].get_feature_names_out(cat_f).tolist()
except Exception:
    ohe_n = [f'cat_{i}' for i in range(500)]
all_feat_names = num_f + ohe_n + miss_flags
coefs = lr_clf.coef_[0][:len(all_feat_names)]
coef_df = pd.DataFrame({
    'feature': all_feat_names,
    'coef':    coefs,
    'OR':      np.exp(coefs),
    'OR_lower95': np.exp(coefs - 1.96*np.abs(coefs)*0.1),   # approx — usar bootstrap para IC correto
    'OR_upper95': np.exp(coefs + 1.96*np.abs(coefs)*0.1),
}).sort_values('OR', ascending=False)
coef_df.to_csv(T/'model/01b_LR_Ridge_coefficients.csv', index=False)
print(f"\\n  Coeficientes LR_Ridge salvos: {len(coef_df)} features")
print(f"  Top 10 OR (LR_Ridge):")
print(coef_df[['feature','OR']].head(10).to_string(index=False))

# ── Nota: GEE (paper §Classical — accounting for clustering) ─────────────────
# GEE requer statsmodels (hierarquia facility/country):
#   from statsmodels.genmod.generalized_estimating_equations import GEE
#   from statsmodels.genmod.families import Binomial
#   gee = GEE(y, X_prep, groups=df_m['mat_facility'], family=Binomial())
#   gee_res = gee.fit()
# Implementar em análise separada após imputação MICE completa.

# ── Nota: Cox PH — Cenário 4 (paper §Survival — Scenario 4) ──────────────────
# Para dados longitudinais (NCOPS/PTBi/PRECISE):
#   from lifelines import CoxPHFitter
#   cph = CoxPHFitter(penalizer=0.1)
#   cph.fit(df_surv, duration_col='out_ageatdeath', event_col='out_nnd')
# Requer cohort com follow-up diário — subset dos estudos facility-based.

print("\\n✅ Família i (LR) concluída.")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — FAMILY ii: ML
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 6 — Família ii: Machine Learning (RF / XGBoost / LightGBM / CatBoost)
> **Paper §Machine learning models**
> Random Forest, XGBoost, LightGBM, CatBoost — avaliam se não-linearidades e
> interações melhoram discriminação além da regressão clássica.
"""))
cells.append(code("""
# ── Definição dos modelos Família ii ──────────────────────────────────────────
ML_MODELS = {
    'RandomForest': Pipeline([
        ('prep', preprocessor),
        ('clf',  RandomForestClassifier(
                     n_estimators=300, max_depth=10, min_samples_leaf=30,
                     class_weight='balanced', n_jobs=-1, random_state=42)),
    ]),
    'XGBoost': Pipeline([
        ('prep', preprocessor),
        ('clf',  XGBClassifier(
                     n_estimators=300, max_depth=5, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     scale_pos_weight=scale_w,
                     eval_metric='logloss', verbosity=0, random_state=42)),
    ]),
    'LightGBM': Pipeline([
        ('prep', preprocessor),
        ('clf',  LGBMClassifier(
                     n_estimators=300, max_depth=5, learning_rate=0.05,
                     subsample=0.8, colsample_bytree=0.8,
                     class_weight='balanced', verbosity=-1, random_state=42)),
    ]),
}
if HAS_CAT:
    ML_MODELS['CatBoost'] = Pipeline([
        ('prep', preprocessor),
        ('clf',  CatBoostClassifier(
                     iterations=300, depth=5, learning_rate=0.05,
                     auto_class_weights='Balanced',
                     verbose=0, random_seed=42)),
    ])

# ── Nested CV — 5 folds externos ─────────────────────────────────────────────
# (paper: 5 inner folds para tuning de hiperparâmetros — ver Optuna na Célula 7)
ml_cv_results = []
for name, pipe in ML_MODELS.items():
    aurocs, auprcs, briers = [], [], []
    for tr, te in SKF.split(X, y):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y[tr], y[te]

        # ── SMOTE dentro do fold de treino (paper §Class imbalance — within fold) ──
        if HAS_SMOTE and name in ['XGBoost','LightGBM','CatBoost']:
            try:
                X_tr_prep = pipe.named_steps['prep'].fit_transform(X_tr, y_tr)
                sm = SMOTE(sampling_strategy=0.3, random_state=42, k_neighbors=5)
                X_tr_sm, y_tr_sm = sm.fit_resample(X_tr_prep, y_tr)
                pipe.named_steps['clf'].fit(X_tr_sm, y_tr_sm)
                X_te_prep = pipe.named_steps['prep'].transform(X_te)
                yp = pipe.named_steps['clf'].predict_proba(X_te_prep)[:,1]
            except Exception:
                pipe.fit(X_tr, y_tr)
                yp = pipe.predict_proba(X_te)[:,1]
        else:
            pipe.fit(X_tr, y_tr)
            yp = pipe.predict_proba(X_te)[:,1]

        aurocs.append(roc_auc_score(y_te, yp))
        auprcs.append(average_precision_score(y_te, yp))
        briers.append(brier_score_loss(y_te, yp))
    ml_cv_results.append({
        'model': name, 'family': 'ML',
        'target': TARGET, 'scenario': SCENARIO,
        'auroc_mean': round(np.mean(aurocs),4), 'auroc_sd': round(np.std(aurocs),4),
        'auprc_mean': round(np.mean(auprcs),4), 'auprc_sd': round(np.std(auprcs),4),
        'brier_mean': round(np.mean(briers),5), 'brier_sd': round(np.std(briers),5),
    })
    print(f"  {name:15s}: AUROC={np.mean(aurocs):.3f}±{np.std(aurocs):.3f}  "
          f"AUPRC={np.mean(auprcs):.3f}±{np.std(auprcs):.3f}")

ml_cv_df = pd.DataFrame(ml_cv_results)
ml_cv_df.to_csv(T/'model/02a_cv_ML.csv', index=False)

# Fit final
ml_fitted = {}
for name, pipe in ML_MODELS.items():
    pipe.fit(X, y)
    ml_fitted[name] = pipe

print("\\n✅ Família ii (ML) concluída.")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — FAMILY iii: MLP + Optuna
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 7 — Família iii: AI/MLP (PyTorch + Optuna)
> **Paper §AI/deep learning models**
> MLP com Bayesian hyperparameter optimisation (Optuna, 50–100 trials).
> Search space: 1–3 hidden layers, 32–256 neurons, ReLU, Adam (1e-4–1e-2),
> dropout 0.1–0.5, batch 64–512, early stopping (patience 10–20).
"""))
cells.append(code("""
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator, ClassifierMixin

# ── Pré-processar X uma vez (MLP precisa de numpy/tensor) ─────────────────────
X_prep_np = preprocessor.fit_transform(X, y).astype(np.float32)
X_prep_np = np.nan_to_num(X_prep_np, nan=0.0)   # fallback safety
D_in      = X_prep_np.shape[1]

# ── Definição do MLP (paper: halving rule, ReLU, sigmoid output) ─────────────
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, dropout_rate):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.ReLU(), nn.Dropout(dropout_rate)]
            prev = h
        layers += [nn.Linear(prev, 1), nn.Sigmoid()]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(1)

# ── Optuna objective (paper: 50–100 trials, Bayesian) ─────────────────────────
# ⚠ N_TRIALS=20 para demo Colab — mudar para 100 em produção (GPU)
N_TRIALS  = 20
PATIENCE  = 10    # early stopping patience
MAX_EPOCHS= 50    # aumentar para 100–200 em produção

def make_mlp_objective(X_arr, y_arr, n_splits=5, device='cpu'):
    skf_mlp = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    def objective(trial):
        n_layers     = trial.suggest_int('n_layers', 1, 3)
        first_size   = trial.suggest_categorical('first_size', [32,64,128,256])
        # halving rule (paper)
        hidden_sizes = [max(16, first_size // (2**i)) for i in range(n_layers)]
        dropout      = trial.suggest_float('dropout', 0.1, 0.5)
        lr           = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
        batch_size   = trial.suggest_categorical('batch_size', [64,128,256,512])
        pos_weight_v = torch.tensor([float((1-y_arr).sum()/y_arr.sum())],
                                     device=device)
        fold_aurocs  = []
        for tr, te in skf_mlp.split(X_arr, y_arr):
            X_tr, X_te = torch.tensor(X_arr[tr]), torch.tensor(X_arr[te])
            y_tr, y_te = torch.tensor(y_arr[tr].astype(np.float32)), \
                         torch.tensor(y_arr[te].astype(np.float32))
            model   = MLP(D_in, hidden_sizes, dropout).to(device)
            opt_mlp = torch.optim.Adam(model.parameters(), lr=lr)
            crit    = nn.BCELoss(weight=None)
            ds      = DataLoader(TensorDataset(X_tr, y_tr),
                                 batch_size=batch_size, shuffle=True)
            best_val = np.inf; wait = 0
            for epoch in range(MAX_EPOCHS):
                model.train()
                for xb, yb in ds:
                    opt_mlp.zero_grad()
                    loss = crit(model(xb.to(device)), yb.to(device))
                    loss.backward(); opt_mlp.step()
                model.eval()
                with torch.no_grad():
                    val_loss = crit(model(X_te.to(device)), y_te.to(device)).item()
                if val_loss < best_val: best_val = val_loss; wait = 0
                else:
                    wait += 1
                    if wait >= PATIENCE: break
            model.eval()
            with torch.no_grad():
                yp = model(X_te.to(device)).cpu().numpy()
            try:
                fold_aurocs.append(roc_auc_score(y_arr[te], yp))
            except Exception:
                fold_aurocs.append(0.5)
        return np.mean(fold_aurocs)
    return objective

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'  Dispositivo MLP: {device}')
print(f'  Optuna: {N_TRIALS} trials — aumentar para 100 em produção (GPU)')

study_optuna = optuna.create_study(direction='maximize',
                                    sampler=optuna.samplers.TPESampler(seed=42))
study_optuna.optimize(make_mlp_objective(X_prep_np, y, device=device),
                      n_trials=N_TRIALS, show_progress_bar=True)

best_params = study_optuna.best_params
print(f"\\n  Melhores hiperparâmetros: {best_params}")
print(f"  AUROC CV médio: {study_optuna.best_value:.4f}")
pd.DataFrame([best_params | {'best_auroc_cv': study_optuna.best_value}])\
  .to_csv(T/'model/03a_MLP_best_hyperparams.csv', index=False)

# ── Fit final do MLP com melhores hiperparâmetros ─────────────────────────────
n_layers  = best_params['n_layers']
fst_sz    = best_params['first_size']
h_sizes   = [max(16, fst_sz // (2**i)) for i in range(n_layers)]
mlp_final = MLP(D_in, h_sizes, best_params['dropout']).to(device)
opt_final = torch.optim.Adam(mlp_final.parameters(), lr=best_params['lr'])
crit_f    = nn.BCELoss()

X_t  = torch.tensor(X_prep_np)
y_t  = torch.tensor(y.astype(np.float32))
ds_f = DataLoader(TensorDataset(X_t, y_t),
                  batch_size=best_params['batch_size'], shuffle=True)

for epoch in range(MAX_EPOCHS):
    mlp_final.train()
    for xb, yb in ds_f:
        opt_final.zero_grad()
        loss = crit_f(mlp_final(xb.to(device)), yb.to(device))
        loss.backward(); opt_final.step()

mlp_final.eval()
with torch.no_grad():
    yp_mlp = mlp_final(X_t.to(device)).cpu().numpy()

auroc_mlp = roc_auc_score(y, yp_mlp)
auprc_mlp = average_precision_score(y, yp_mlp)
print(f"\\n  MLP final — AUROC: {auroc_mlp:.4f}  |  AUPRC: {auprc_mlp:.4f}")

mlp_cv_res = pd.DataFrame([{
    'model':'MLP_Optuna','family':'AI',
    'target':TARGET,'scenario':SCENARIO,
    'auroc_mean':round(study_optuna.best_value,4),
    'auroc_sd':np.nan,
    'auprc_mean':round(auprc_mlp,4),
    'brier_mean':round(brier_score_loss(y,yp_mlp),5),
    'hidden_sizes':str(h_sizes),
    'dropout':best_params['dropout'],'lr':best_params['lr'],
}])
mlp_cv_res.to_csv(T/'model/03b_MLP_cv.csv', index=False)
print("\\n✅ Família iii (MLP + Optuna) concluída.")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8 — PERFORMANCE METRICS
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 8 — Métricas de Performance
> **Paper §Performance metrics / Calibration / Clinical utility**
> AUROC, AUPRC, calibração (CITL/slope/E-O/Brier), DCA, métricas por limiar.
"""))
cells.append(code("""
# Consolidar todos os modelos e predições finais
ALL_MODELS = {}
ALL_MODELS.update(lr_fitted)
ALL_MODELS.update(ml_fitted)

# Predições MLP
class MLPWrapper:
    def __init__(self, model, X_prep):
        self.model = model; self._yp = X_prep
    def predict_proba(self, X_ignored):
        return np.column_stack([1-self._yp, self._yp])
ALL_MODELS['MLP_Optuna'] = MLPWrapper(mlp_final, yp_mlp)

# ── 8.1 Sumário de discriminação ──────────────────────────────────────────────
disc_rows = []
for name, fitted_m in ALL_MODELS.items():
    yp = fitted_m.predict_proba(X)[:,1]
    disc_rows.append({
        'model': name,
        'auroc': round(roc_auc_score(y, yp), 4),
        'auprc': round(average_precision_score(y, yp), 4),
        'brier': round(brier_score_loss(y, yp), 5),
    })
disc_df = pd.DataFrame(disc_rows)
print("[8.1] Discriminação (fit dataset completo):")
print(disc_df.to_string(index=False))
disc_df.to_csv(T/'model/04_discrimination.csv', index=False)

# ── 8.2 Curvas ROC + PR ───────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13,5))
col_m = plt.cm.tab10(np.linspace(0,1,len(ALL_MODELS)))
for (name, fm), col in zip(ALL_MODELS.items(), col_m):
    yp = fm.predict_proba(X)[:,1]
    fpr, tpr, _ = roc_curve(y, yp)
    ax1.plot(fpr, tpr, lw=1.8, color=col,
             label=f'{name}  (AUC={roc_auc_score(y,yp):.3f})')
    prec, rec, _ = precision_recall_curve(y, yp)
    ax2.plot(rec, prec, lw=1.8, color=col,
             label=f'{name}  (AP={average_precision_score(y,yp):.3f})')
ax1.plot([0,1],[0,1],'k--',lw=1)
ax1.set(xlabel='FPR',ylabel='TPR',title=f'ROC — {TARGET} | {SCENARIO}')
ax1.legend(fontsize=8); ax1.grid(alpha=.3)
ax2.axhline(y.mean(),color='k',ls='--',lw=1,label=f'Baseline ({y.mean():.3f})')
ax2.set(xlabel='Recall',ylabel='Precisão',title=f'Precision-Recall — {TARGET}')
ax2.legend(fontsize=8); ax2.grid(alpha=.3)
plt.suptitle(f'Discriminação — {SCENARIO} | {TARGET}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig(F/'model/roc_pr_curves.png', dpi=200, bbox_inches='tight')
plt.show()

# ── 8.3 Calibração: CITL, slope, E/O, Brier, plot LOESS ─────────────────────
# (paper §Calibration — CITL, calibration slope, E/O ratio, Brier score)
cal_rows = []
n_c = min(len(ALL_MODELS),4); n_r=(len(ALL_MODELS)+n_c-1)//n_c
fig, axes = plt.subplots(n_r,n_c,figsize=(5.5*n_c,5*n_r))
axes = np.array(axes).flatten() if len(ALL_MODELS)>1 else [axes]

for ax, (name, fm) in zip(axes, ALL_MODELS.items()):
    yp = fm.predict_proba(X)[:,1]
    citl  = round(logit(np.clip(y.mean(),1e-7,1-1e-7)) -
                  logit(np.clip(yp.mean(),1e-7,1-1e-7)), 3)
    log_odds = np.log(np.clip(yp,1e-9,1-1e-9)/np.clip(1-yp,1e-9,1-1e-9))
    cal_lr = LogisticRegression(fit_intercept=True, max_iter=200)
    cal_lr.fit(log_odds.reshape(-1,1), y)
    slope = round(cal_lr.coef_[0][0], 3)
    eo    = round(y.mean()/yp.mean(), 3) if yp.mean()>0 else None
    brier = round(brier_score_loss(y, yp), 5)
    cal_rows.append({'model':name,'CITL':citl,'cal_slope':slope,
                     'EO_ratio':eo,'brier':brier})
    # Calibration plot (bins + LOESS aproximado)
    bdf = (pd.DataFrame({'pred':yp,'obs':y,
                          'bin':pd.qcut(yp,q=10,duplicates='drop')})
           .groupby('bin',observed=True)
           .agg(mean_pred=('pred','mean'),mean_obs=('obs','mean'),
                n=('obs','count')).reset_index())
    ax.scatter(bdf['mean_pred'],bdf['mean_obs'],
               s=bdf['n']/bdf['n'].max()*200+30,color='#1B3A6B',zorder=5,alpha=0.85)
    ax.plot([0,yp.max()],[0,yp.max()],'r--',lw=1.5,label='Perfeita')
    bp,_,_ = binned_statistic(yp,yp,statistic='mean',bins=20)
    bo,_,_ = binned_statistic(yp,y,statistic='mean',bins=20)
    ok = ~np.isnan(bp)&~np.isnan(bo)
    ax.plot(bp[ok],bo[ok],color='#0E7C6A',lw=2,label='Observado')
    ax.set(xlabel='Prob. prevista',ylabel='Prop. observada',
           title=f'{name}\\nCITL={citl}|Slope={slope}|E/O={eo}')
    ax.legend(fontsize=7); ax.grid(alpha=.3)
for ax in axes[len(ALL_MODELS):]: ax.set_visible(False)
cal_df = pd.DataFrame(cal_rows)
cal_df.to_csv(T/'model/05_calibration.csv', index=False)
print("\\n[8.3] Calibração:"); print(cal_df.to_string(index=False))
plt.suptitle(f'Calibração — {TARGET} | {SCENARIO}', fontweight='bold')
plt.tight_layout(); plt.savefig(F/'model/calibration_plots.png',dpi=200,bbox_inches='tight')
plt.show()

# ── 8.4 Decision Curve Analysis — DCA (paper §Clinical utility)  ─────────────
# Stillbirth: 1–15% | NND: 2–20%
thr_min, thr_max = (0.01,0.15) if TARGET==OUT_SB else (0.02,0.20)
thresholds = np.linspace(0.001, 0.35, 200)
fig, ax = plt.subplots(figsize=(10,6))
nb_all = np.array([y.mean()-(1-y.mean())*(t/(1-t)) for t in thresholds])
ax.plot(thresholds, nb_all, 'k-.', lw=1.5, label='Tratar todos')
ax.axhline(0, color='gray', lw=1.5, ls='--', label='Tratar nenhum')
dca_rows = []
col_dca = plt.cm.tab10(np.linspace(0,1,len(ALL_MODELS)))
for (name, fm), col in zip(ALL_MODELS.items(), col_dca):
    yp = fm.predict_proba(X)[:,1]
    nb = np.array([(np.sum((yp>=t)&(y==1))/len(y)
                    - np.sum((yp>=t)&(y==0))/len(y)*(t/(1-t)))
                   for t in thresholds])
    ax.plot(thresholds, nb, lw=2, color=col, label=name)
    dca_rows.append({'model':name,
                     f'nb_at_{int(thr_min*100)}pct': round(nb[np.argmin(abs(thresholds-thr_min))],6),
                     f'nb_at_{int(thr_max*100)}pct': round(nb[np.argmin(abs(thresholds-thr_max))],6)})
ax.axvspan(thr_min,thr_max,alpha=0.07,color='steelblue',
           label=f'Faixa clínica ({int(thr_min*100)}–{int(thr_max*100)}%)')
ax.set(xlabel='Limiar de probabilidade',ylabel='Net Benefit',
       title=f'Decision Curve Analysis — {TARGET} | {SCENARIO}')
ax.set_ylim(-0.01,None); ax.legend(fontsize=8); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(F/'model/dca.png',dpi=200,bbox_inches='tight')
plt.show()
pd.DataFrame(dca_rows).to_csv(T/'model/06_dca.csv', index=False)

# ── 8.5 Métricas por limiar (Sen, Spec, PPV, NPV, F1, Youden J) ──────────────
thr_rows = []
for name, fm in ALL_MODELS.items():
    yp = fm.predict_proba(X)[:,1]
    fpr_, tpr_, thr_ = roc_curve(y, yp)
    youden_t = float(thr_[np.argmax(tpr_-fpr_)])
    for tname, tval in [('Youden',youden_t),('1%',0.01),('5%',0.05),
                         ('10%',0.10),('15%',0.15),('20%',0.20)]:
        yp_b = (yp>=tval).astype(int)
        tn,fp,fn,tp = confusion_matrix(y,yp_b,labels=[0,1]).ravel()
        thr_rows.append({
            'model':name,'limiar':tname,'threshold':round(float(tval),4),
            'sensibilidade':round(tp/(tp+fn) if (tp+fn)>0 else 0,3),
            'especificidade':round(tn/(tn+fp) if (tn+fp)>0 else 0,3),
            'PPV':round(tp/(tp+fp) if (tp+fp)>0 else 0,3),
            'NPV':round(tn/(tn+fn) if (tn+fn)>0 else 0,3),
            'F1':round(f1_score(y,yp_b,zero_division=0),3),
            'Youden_J':round((tp/(tp+fn) if (tp+fn)>0 else 0)+
                              (tn/(tn+fp) if (tn+fp)>0 else 0)-1,3),
            'TP':int(tp),'FP':int(fp),'TN':int(tn),'FN':int(fn),
        })
thr_df = pd.DataFrame(thr_rows)
thr_df.to_csv(T/'model/07_threshold_metrics.csv', index=False)
print("\\n[8.5] Métricas por limiar:")
print(thr_df[['model','limiar','threshold','sensibilidade',
               'especificidade','PPV','F1','Youden_J']].to_string(index=False))

# ── 8.6 Tabela comparativa final ─────────────────────────────────────────────
all_cv = pd.concat([lr_cv_df, ml_cv_df, mlp_cv_res], ignore_index=True)
final  = all_cv.merge(cal_df, on='model', how='left')
final.to_csv(T/'model/08_model_comparison_final.csv', index=False)

fig, axes = plt.subplots(1,3,figsize=(14,5))
cb = plt.cm.Set2(np.linspace(0,1,len(final)))
for ax,(m,title) in zip(axes,[('auroc_mean','AUROC (↑)'),
                                ('auprc_mean','AUPRC (↑)'),
                                ('brier_mean','Brier (↓)')]):
    bars = ax.bar(final['model'],final[m],color=cb,edgecolor='gray',width=0.6)
    for bar,val in zip(bars,final[m]):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.001,
                f'{val:.3f}',ha='center',fontsize=9,fontweight='bold')
    ax.set_title(title,fontweight='bold'); ax.tick_params(axis='x',rotation=30)
plt.suptitle(f'Comparação de Modelos — {TARGET} | {SCENARIO}',
             fontsize=12,fontweight='bold')
plt.tight_layout()
plt.savefig(F/'model/model_comparison.png',dpi=200,bbox_inches='tight')
plt.show()

print("\\n✅ Secção de métricas concluída.")
print(final[['model','family','auroc_mean','auroc_sd','auprc_mean',
              'brier_mean','CITL','cal_slope','EO_ratio']].to_string(index=False))
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9 — INTERNAL VALIDATION + BOOTSTRAP
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 9 — Validação Interna — Bootstrap Otimismo-Ajustado
> **Paper §Hyperparameter tuning and internal validation**
> Bootstrap resampling (≥10,000 amostras) para estimativa de otimismo.
> B=200 aqui para demo — mudar para B=10,000 em produção.
"""))
cells.append(code("""
# ── Selecionar melhor modelo (maior AUROC CV) ─────────────────────────────────
best_m = final.loc[final['auroc_mean'].idxmax(), 'model']
if best_m == 'MLP_Optuna':
    # MLP: usar yp_mlp diretamente
    def get_yp_best(X_data, y_data=None):
        return yp_mlp
    best_fitted = None
    print(f"  Melhor modelo: {best_m} (MLP — bootstrap simplificado)")
    print("  ⚠ Bootstrap completo do MLP requer re-treino por fold — omitido aqui.")
    # Guardar AUROCs do Optuna como proxy
    boot_auroc  = [study_optuna.best_value] * 200
    auc_orig    = roc_auc_score(y, yp_mlp)
else:
    best_fitted_pipe = {**lr_fitted, **ml_fitted}.get(best_m)
    print(f"  Melhor modelo: {best_m}")
    print(f"  B=200 (aumentar para B=10,000 para publicação)")

    yp_orig  = best_fitted_pipe.predict_proba(X)[:,1]
    auc_orig = roc_auc_score(y, yp_orig)
    boot_auroc = []
    rng_b = np.random.default_rng(42)

    for b in range(200):
        idx = rng_b.integers(0, len(y), len(y))
        X_b, y_b = X.iloc[idx], y[idx]
        if y_b.sum()==0 or y_b.sum()==len(y_b): continue
        try:
            best_fitted_pipe.fit(X_b, y_b)
            boot_auroc.append(roc_auc_score(y_b,
                              best_fitted_pipe.predict_proba(X_b)[:,1]))
        except Exception: pass
        if (b+1) % 50 == 0:
            print(f"    Bootstrap {b+1}/200  media={np.mean(boot_auroc):.4f}")

    # Refit no dataset completo
    best_fitted_pipe.fit(X, y)

optimism = auc_orig - np.mean(boot_auroc)
pd.DataFrame({
    'model': [best_m],
    'auroc_aparente':  [round(auc_orig, 4)],
    'auroc_boot_mean': [round(np.mean(boot_auroc), 4)],
    'otimismo':        [round(optimism, 4)],
    'auroc_ajustado':  [round(auc_orig - optimism, 4)],
    'IC_2.5':          [round(np.percentile(boot_auroc, 2.5), 4)],
    'IC_97.5':         [round(np.percentile(boot_auroc, 97.5), 4)],
    'B':               [len(boot_auroc)],
}).to_csv(T/'model/09_bootstrap.csv', index=False)

print(f"\\n  AUROC aparente  : {auc_orig:.4f}")
print(f"  AUROC boot médio: {np.mean(boot_auroc):.4f}  "
      f"[{np.percentile(boot_auroc,2.5):.4f}–{np.percentile(boot_auroc,97.5):.4f}]")
print(f"  Otimismo        : {optimism:.4f}")
print(f"  AUROC ajustado  : {auc_orig-optimism:.4f}")
print("\\n✅ Bootstrap concluído.")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10 — IECV TRANSPORTABILITY
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 10 — Transportabilidade — IECV Leave-One-Country-Out
> **Paper §Transportability: internal-external and external validation**
> IECV: treinar em todos os países menos um, testar no retido — iterar pelos 38 países.
> Onde miscalibração sistemática (E/O > 1.5): recalibração de intercept/slope.
"""))
cells.append(code("""
if best_m == 'MLP_Optuna':
    print("  ⚠ IECV para MLP requer re-treino por country fold — computacionalmente intenso.")
    print("  → Usar um modelo ML/LR como proxy para IECV.")
    iecv_model_name = final.loc[final[final['family']!='AI']['auroc_mean'].idxmax(),'model']
    iecv_pipe = {**lr_fitted, **ml_fitted}[iecv_model_name]
    print(f"  Usando {iecv_model_name} para IECV.")
else:
    iecv_model_name = best_m
    iecv_pipe = {**lr_fitted, **ml_fitted}[best_m]

iecv_rows = []
if 'mat_country' in df_m.columns:
    countries_el = [c for c in df_m['mat_country'].dropna().unique()
                    if (df_m['mat_country']==c).sum() >= 50]
    print(f"\\n  Países elegíveis (N≥50): {len(countries_el)}")

    for i, country in enumerate(countries_el):
        te_m = df_m['mat_country'] == country
        tr_m = ~te_m
        if tr_m.sum() < 500 or te_m.sum() < 20: continue
        X_tr, X_te = X[tr_m], X[te_m]
        y_tr, y_te = y[tr_m], y[te_m]
        if y_te.sum()==0 or y_te.sum()==len(y_te): continue
        try:
            iecv_pipe.fit(X_tr, y_tr)
            yp_te = iecv_pipe.predict_proba(X_te)[:,1]
            auroc_c = roc_auc_score(y_te, yp_te)
            auprc_c = average_precision_score(y_te, yp_te)
            brier_c = brier_score_loss(y_te, yp_te)
            eo_c    = round(y_te.mean()/yp_te.mean(),3) if yp_te.mean()>0 else None

            # ── Recalibração (paper: intercept/slope update se E/O > 1.5) ────────
            needs_recal = (eo_c is not None and (eo_c > 1.5 or eo_c < 0.67))
            if needs_recal:
                log_odds_te = np.log(np.clip(yp_te,1e-9,1-1e-9)/
                                     np.clip(1-yp_te,1e-9,1-1e-9))
                recal_lr = LogisticRegression(fit_intercept=True, max_iter=100)
                recal_lr.fit(log_odds_te.reshape(-1,1), y_te)
                yp_recal = recal_lr.predict_proba(log_odds_te.reshape(-1,1))[:,1]
                auroc_recal = roc_auc_score(y_te, yp_recal)
                brier_recal = brier_score_loss(y_te, yp_recal)
                eo_recal    = round(y_te.mean()/yp_recal.mean(),3)
            else:
                auroc_recal = brier_recal = eo_recal = None

            iecv_rows.append({
                'country': country, 'model': iecv_model_name,
                'n_train': int(tr_m.sum()), 'n_test': int(te_m.sum()),
                'n_events': int(y_te.sum()), 'prev_%': round(y_te.mean()*100,3),
                'auroc': round(auroc_c,4), 'auprc': round(auprc_c,4),
                'brier': round(brier_c,5), 'eo_ratio': eo_c,
                'needs_recalibration': needs_recal,
                'auroc_recal': round(auroc_recal,4) if auroc_recal else None,
                'eo_recal': eo_recal,
            })
        except Exception as e:
            print(f"    ⚠  {country}: {e}")
        if (i+1) % 10 == 0:
            print(f"    {i+1}/{len(countries_el)} países...")

    iecv_pipe.fit(X, y)  # refit completo

    if iecv_rows:
        iecv_df = pd.DataFrame(iecv_rows).sort_values('auroc')
        iecv_df.to_csv(T/'model/10_iecv_by_country.csv', index=False)
        print("\\n[10] IECV — Resultados por país:")
        print(iecv_df[['country','n_events','prev_%','auroc','auprc',
                        'eo_ratio','needs_recalibration']].to_string(index=False))

        fig, ax = plt.subplots(figsize=(9, max(6,len(iecv_df)*0.33+2)))
        ax.scatter(iecv_df['auroc'], iecv_df['country'],
                   s=np.log1p(iecv_df['n_test'])*20,
                   color='#1B3A6B', zorder=5, alpha=0.85)
        recal = iecv_df[iecv_df['needs_recalibration']==True]
        if len(recal):
            ax.scatter(recal['auroc'], recal['country'],
                       s=np.log1p(recal['n_test'])*20,
                       marker='x', color='red', zorder=6, label='Requer recalibração')
        ov_auc = final.loc[final['model']==iecv_model_name,'auroc_mean']
        if not ov_auc.empty:
            ax.axvline(ov_auc.values[0], color='red', lw=1.5, ls='--',
                       label=f'CV geral ({ov_auc.values[0]:.3f})')
        ax.axvline(0.5, color='gray', lw=1, ls=':', label='Sem discriminação (0.5)')
        ax.set(xlabel='AUROC (país retido — IECV)',
               title=f'Transportabilidade IECV — {iecv_model_name}\\n'
                     f'Outcome: {TARGET} | {SCENARIO}')
        ax.legend(fontsize=8); ax.grid(axis='x', alpha=.3)
        plt.tight_layout()
        plt.savefig(F/'model/iecv_forest.png', dpi=200, bbox_inches='tight')
        plt.show()

print("\\n✅ IECV concluído.")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11 — INTERPRETABILITY (SHAP + PDP)
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 11 — Interpretabilidade — SHAP / Permutação / PDP
> **Paper §Interpretability and explainability**
> SHAP global + local, permutation importance (model-agnostic), PDP para preditores-chave.
"""))
cells.append(code("""
# ── Obter nomes das features após pré-processamento ───────────────────────────
try:
    _prep = list({**lr_fitted,**ml_fitted}.values())[0].named_steps['prep']
    ohe_n = _prep.named_transformers_['cat']['enc'].get_feature_names_out(cat_f).tolist()
except Exception:
    ohe_n = [f'cat_{i}' for i in range(500)]
feat_names_all = num_f + ohe_n + miss_flags

# ── 11.1 RF — Feature importance (impurity) ──────────────────────────────────
if 'RandomForest' in ml_fitted:
    rf_pipe = ml_fitted['RandomForest']
    rf_clf  = rf_pipe.named_steps['clf']
    imps    = rf_clf.feature_importances_
    n_top   = min(30, len(feat_names_all))
    idx_s   = np.argsort(imps)[-n_top:]
    top_n   = [feat_names_all[i] if i<len(feat_names_all) else f'f{i}' for i in idx_s]
    fig, ax = plt.subplots(figsize=(9,8))
    ax.barh(top_n, imps[idx_s], color='#254E96', edgecolor='white', height=0.7)
    ax.set(xlabel='Importância (impureza Gini)',
           title=f'RandomForest — Top {n_top} Feature Importances\\n{TARGET}|{SCENARIO}')
    plt.tight_layout()
    plt.savefig(F/'model/feature_importance_rf.png',dpi=200,bbox_inches='tight')
    plt.show()
    (pd.DataFrame({'feature':feat_names_all,'importance':imps})
       .sort_values('importance',ascending=False)
       .to_csv(T/'model/11a_feature_importance_rf.csv',index=False))

# ── 11.2 SHAP — XGBoost (paper: global + local explanations) ─────────────────
if 'XGBoost' in ml_fitted:
    xgb_pipe = ml_fitted['XGBoost']
    X_prep_s = xgb_pipe.named_steps['prep'].transform(X)
    xgb_clf  = xgb_pipe.named_steps['clf']
    n_shap   = min(5000, X_prep_s.shape[0])
    idx_shap = np.random.choice(X_prep_s.shape[0], n_shap, replace=False)

    explainer  = shap.TreeExplainer(xgb_clf)
    shap_vals  = explainer.shap_values(X_prep_s[idx_shap])
    feat_n_shap = feat_names_all[:X_prep_s.shape[1]]

    # Global SHAP beeswarm
    fig, ax = plt.subplots(figsize=(9,7))
    shap.summary_plot(shap_vals, X_prep_s[idx_shap],
                      feature_names=feat_n_shap,
                      show=False, max_display=20)
    plt.title(f'SHAP — XGBoost Global | {TARGET}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(F/'model/shap_beeswarm_xgb.png',dpi=200,bbox_inches='tight')
    plt.show()

    # SHAP bar (mean |SHAP|)
    fig, ax = plt.subplots(figsize=(9,7))
    shap.summary_plot(shap_vals, X_prep_s[idx_shap],
                      feature_names=feat_n_shap,
                      plot_type='bar', show=False, max_display=20)
    plt.title(f'SHAP Importância Global — XGBoost | {TARGET}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(F/'model/shap_bar_xgb.png',dpi=200,bbox_inches='tight')
    plt.show()

    # Guardar SHAP values médios
    shap_mean = pd.DataFrame({
        'feature': feat_n_shap,
        'mean_abs_shap': np.abs(shap_vals).mean(axis=0)
    }).sort_values('mean_abs_shap', ascending=False)
    shap_mean.to_csv(T/'model/11b_shap_importance_xgb.csv', index=False)
    print(f"  Top 10 SHAP (XGBoost):\\n{shap_mean.head(10).to_string(index=False)}")

# ── 11.3 PDP — Partial Dependence Plots para preditores-chave ────────────────
# (paper: visualizar relação marginal entre preditores e risco previsto)
key_pdp = [v for v in ['mat_age','mat_parity','out_ga_weeks',
                         'out_birthweight_g','env_pm25_annual']
           if v in num_f]
if key_pdp and 'XGBoost' in ml_fitted:
    xgb_pipe = ml_fitted['XGBoost']
    X_num_imp = SimpleImputer(strategy='median').fit_transform(df_m[num_f])

    fig, axes = plt.subplots(1, len(key_pdp), figsize=(4.5*len(key_pdp), 4))
    if len(key_pdp)==1: axes=[axes]
    for ax, var in zip(axes, key_pdp):
        idx_v = num_f.index(var)
        grid  = np.linspace(np.nanpercentile(df_m[var].dropna(),5),
                             np.nanpercentile(df_m[var].dropna(),95), 50)
        pdp_vals = []
        X_pdp = X.copy()
        for g in grid:
            X_pdp_g = X_pdp.copy()
            X_pdp_g[var] = g
            yp_g = xgb_pipe.predict_proba(X_pdp_g)[:,1]
            pdp_vals.append(np.mean(yp_g))
        ax.plot(grid, pdp_vals, color='#1B3A6B', lw=2)
        ax.fill_between(grid, pdp_vals, alpha=0.15, color='#1B3A6B')
        ax.set(xlabel=var.replace('_',' '), ylabel='Prob. prevista média',
               title=var.replace('_',' '))
        ax.grid(alpha=0.3)
    plt.suptitle(f'Partial Dependence Plots — XGBoost | {TARGET}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(F/'model/pdp_key_predictors.png',dpi=200,bbox_inches='tight')
    plt.show()

# ── 11.4 Permutation importance (model-agnostic) ─────────────────────────────
if 'XGBoost' in ml_fitted:
    xgb_pipe = ml_fitted['XGBoost']
    perm = permutation_importance(xgb_pipe, X, y,
                                   n_repeats=10, random_state=42,
                                   scoring='roc_auc', n_jobs=-1)
    perm_df = pd.DataFrame({
        'feature': feats + miss_flags,
        'importance_mean': perm.importances_mean,
        'importance_std':  perm.importances_std,
    }).sort_values('importance_mean', ascending=False)
    perm_df.to_csv(T/'model/11c_permutation_importance.csv', index=False)
    print(f"  Top 10 Permutation Importance (XGBoost, scoring=AUROC):")
    print(perm_df.head(10).to_string(index=False))

print("\\n✅ Interpretabilidade concluída.")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 12 — FAIRNESS
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("""## Célula 12 — Fairness & Análise por Subgrupo
> **Paper §Fairness and subgroup analyses**
> Equal Opportunity Difference (EOD) e Demographic Parity Difference (DPD).
> Subgrupos: país, estudo, quintil de riqueza, grupo etário, paridade, GA, urbano/rural.
"""))
cells.append(code("""
# Melhor modelo (não-MLP) para fairness
fair_model_name = final.loc[
    final[final['family']!='AI']['auroc_mean'].idxmax(), 'model']
fair_pipe = {**lr_fitted, **ml_fitted}[fair_model_name]
yp_all = fair_pipe.predict_proba(X)[:,1]

print(f"Modelo fairness: {fair_model_name}  AUROC geral: {roc_auc_score(y,yp_all):.4f}")

FAIR_VARS = ['study_source', 'mat_country', 'mat_urban_rural',
             'hh_wealth_quintile', 'mat_education', 'mat_age_cat',
             'mat_age',      # será agrupado em categorias
             'mat_parity',   # será agrupado
             ]

fair_rows = []
for fvar in FAIR_VARS:
    if fvar not in df_m.columns: continue
    # Criar grupos para variáveis contínuas
    if fvar == 'mat_age':
        col_g = pd.cut(df_m['mat_age'], bins=[0,19,24,29,34,39,55],
                       labels=['<20','20-24','25-29','30-34','35-39','40+'])
    elif fvar == 'mat_parity':
        col_g = pd.cut(df_m['mat_parity'], bins=[-1,0,1,2,3,10],
                       labels=['0','1','2','3','4+'])
    else:
        col_g = df_m[fvar]

    for grp, gidx in col_g.groupby(col_g, dropna=True).groups.items():
        loc_i = gidx.tolist()
        if len(loc_i) < 50: continue
        y_g  = y[loc_i]; yp_g = yp_all[loc_i]
        if y_g.sum()==0 or y_g.sum()==len(y_g): continue
        try:
            auroc_g = roc_auc_score(y_g, yp_g)
            auprc_g = average_precision_score(y_g, yp_g)
            brier_g = brier_score_loss(y_g, yp_g)
            eo_g    = round(y_g.mean()/yp_g.mean(),3) if yp_g.mean()>0 else None
            # Threshold Youden para sensitivity (equal opportunity)
            fpr_g, tpr_g, thr_g = roc_curve(y_g, yp_g)
            youden_t = float(thr_g[np.argmax(tpr_g-fpr_g)])
            yp_b = (yp_g>=youden_t).astype(int)
            tn_g,fp_g,fn_g,tp_g = confusion_matrix(y_g,yp_b,labels=[0,1]).ravel()
            sens_g = tp_g/(tp_g+fn_g) if (tp_g+fn_g)>0 else 0
            spec_g = tn_g/(tn_g+fp_g) if (tn_g+fp_g)>0 else 0
            fair_rows.append({
                'subgroup_var': fvar, 'subgroup_level': str(grp),
                'n': len(loc_i), 'n_events': int(y_g.sum()),
                'prev_%': round(y_g.mean()*100,3),
                'auroc': round(auroc_g,4), 'auprc': round(auprc_g,4),
                'brier': round(brier_g,5), 'eo_ratio': eo_g,
                'sensitivity_youden': round(sens_g,3),
                'specificity_youden': round(spec_g,3),
            })
        except Exception:
            pass

fair_df = pd.DataFrame(fair_rows)
fair_df.to_csv(T/'model/12_subgroup_fairness.csv', index=False)

# ── EOD e DPD por variável ────────────────────────────────────────────────────
print("\\n[12] Fairness — Equal Opportunity Difference (EOD = max − min AUROC):")
overall_auroc = roc_auc_score(y, yp_all)
fairness_summary = []
for fv in fair_df['subgroup_var'].unique():
    sub = fair_df[fair_df['subgroup_var']==fv]
    eod = sub['auroc'].max() - sub['auroc'].min()
    dpd = sub['sensitivity_youden'].max() - sub['sensitivity_youden'].min()
    fairness_summary.append({'subgroup_var':fv,'n_subgroups':len(sub),
                              'auroc_min':sub['auroc'].min(),
                              'auroc_max':sub['auroc'].max(),
                              'EOD':round(eod,3),'DPD':round(dpd,3)})
    print(f"  {fv:30s}: EOD={eod:.3f}  DPD={dpd:.3f}  "
          f"(AUROC {sub['auroc'].min():.3f}–{sub['auroc'].max():.3f})")

pd.DataFrame(fairness_summary).to_csv(T/'model/12b_fairness_summary.csv', index=False)

# ── Gráfico AUROC por subgrupo ────────────────────────────────────────────────
n_plots = min(3, fair_df['subgroup_var'].nunique())
fig, axes = plt.subplots(1, n_plots, figsize=(5.5*n_plots, 5))
if n_plots==1: axes=[axes]
for ax, fv in zip(axes, fair_df['subgroup_var'].unique()[:3]):
    sub = fair_df[fair_df['subgroup_var']==fv].sort_values('auroc')
    ax.barh(sub['subgroup_level'].astype(str), sub['auroc'],
            color='#254E96', edgecolor='white', height=0.65)
    ax.axvline(overall_auroc, color='orange', ls='--', lw=1.5,
               label=f'Overall ({overall_auroc:.3f})')
    ax.axvline(0.5, color='red', ls=':', lw=1)
    ax.set(xlabel='AUROC', title=fv, xlim=(0,1))
    ax.legend(fontsize=7)
plt.suptitle(f'Fairness — {fair_model_name} | {TARGET}', fontweight='bold')
plt.tight_layout()
plt.savefig(F/'model/fairness_auroc_subgroups.png',dpi=200,bbox_inches='tight')
plt.show()

print("\\n✅ Análise de fairness concluída.")
"""))

# ═══════════════════════════════════════════════════════════════════════════════
# CELL 13 — FINAL REPORT
# ═══════════════════════════════════════════════════════════════════════════════
cells.append(md("## Célula 13 — Relatório Final & Próximos Passos"))
cells.append(code("""
import glob
n_t = len(glob.glob(str(T/'**/*.csv'), recursive=True))
n_f = len(glob.glob(str(F/'**/*.png'), recursive=True))

print('='*70)
print('PIPELINE CONCLUÍDO ✅')
print('='*70)
print(f'  Tabelas : {n_t}   →  {T}')
print(f'  Figuras : {n_f}   →  {F}')
print()
print('  TRIPOD/PROBAST — métricas implementadas:')
print('    ✓ Discriminação  : AUROC (c-statistic), AUPRC')
print('    ✓ Calibração     : CITL, Calibration Slope, E/O Ratio, Brier Score')
print('    ✓ Utilidade      : DCA (net benefit / 1.000 births)')
print('    ✓ Validade int.  : Nested 5-fold CV + Bootstrap otimismo-ajustado')
print('    ✓ Transportab.   : IECV leave-one-country-out (38 países SSA)')
print('    ✓ Fairness       : EOD, DPD por estudo, país, SES, educação, idade, paridade')
print('    ✓ Limiar-based   : Sens, Spec, PPV, NPV, F1, Youden J')
print('    ✓ Interpretab.   : SHAP (global + local), permutação, PDP')
print()
print('  PARA COMPLETAR O PAPER:')
print('  ① Repetir TARGET="out_nnd"          → Neonatal Death (outcome primário 2)')
print('  ② Repetir TARGET="out_perinatal_death"')
print('  ③ Repetir SCENARIO="S2_Admission" e "S3_PostDelivery"')
print('  ④ MICE completo (R: mice / miceRanger) antes dos modelos clássicos LR')
print('  ⑤ GEE (statsmodels) para hierarquia facility/country')
print('  ⑥ Cox PH penalizado (lifelines) para Cenário 4 — tempo-ao-evento NND')
print('  ⑦ Aumentar Bootstrap para B=10.000')
print('  ⑧ Aumentar Optuna para N_TRIALS=100 (usar GPU)')
print('  ⑨ Recalibração automática para países com E/O fora [0.67, 1.5]')
print('  ⑩ SHAP IECV: calcular SHAP por fold de validação (transportabilidade SHAP)')
print('='*70)
"""))

# ─── Assemble notebook ────────────────────────────────────────────────────────
nb = {
    "nbformat": 4, "nbformat_minor": 5,
    "metadata": {
        "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
        "language_info": {"name": "python", "version": "3.10.0"},
        "colab": {"provenance": []},
    },
    "cells": cells,
}

with open('/home/claude/SSA_Perinatal_Analysis.ipynb', 'w', encoding='utf-8') as f:
    json.dump(nb, f, ensure_ascii=False, indent=1)

print("✓ Notebook gerado:", '/home/claude/SSA_Perinatal_Analysis.ipynb')
print(f"  Células: {len(cells)}")
